# 09 - Markdown Section Parser

在建立 RAG 系統之前，必須先把原始文件轉成結構化的可索引格式。
本節實作一個**純 Python**的 Markdown parser，不依賴 LLM，輸出 JSONL chunks。

目標：把每份 CRM 會議紀錄拆成以下結構：

```json
{
  "doc_id": "meeting_001_晨星科技_2026-05-08",
  "customer": "晨星科技股份有限公司",
  "meeting_date": "2026-05-08",
  "recorder": "王雅婷",
  "section": "風險評估",
  "content": "..."
}
```

每個 `##` 段落（section）獨立成一筆 chunk，metadata 從 `## 基本資訊` 的 Markdown table 萃取。

In [1]:
import json
import re
import sys
from dataclasses import asdict, dataclass
from pathlib import Path

_cwd = Path().resolve()
PROJECT_ROOT = _cwd if (_cwd / 'data').exists() else _cwd.parent
EXAMPLES_DIR = PROJECT_ROOT / 'examples'
CRM_DIR      = PROJECT_ROOT / 'data' / 'crm_notes'

if str(EXAMPLES_DIR) not in sys.path:
    sys.path.insert(0, str(EXAMPLES_DIR))

from utils.tools import FileTools
_crm = FileTools(CRM_DIR)

print(f'Project root: {PROJECT_ROOT}')
print(f'CRM 檔案: {_crm.list_files()}')

Project root: /home/mistin/research/agentic-rag-lab
CRM 檔案: ['meeting_001_晨星科技_2026-05-08.md', 'meeting_002_鴻圖零售_2026-05-15.md', 'meeting_003_晨星科技_2026-05-22.md', 'meeting_004_新星金融科技_2026-05-28.md', 'meeting_005_晨星科技_2026-06-05.md']


## 1 · Metadata 萃取（從 Markdown Table）

CRM 文件沒有 YAML frontmatter，metadata 嵌在 `## 基本資訊` 的 Markdown table 中，格式如下：

```markdown
| 欄位 | 內容 |
|------|------|
| **客戶名稱** | 晨星科技股份有限公司 |
| **會議日期** | 2026-05-08 |
```

用正規表達式解析「粗體 key → value」的 table row 格式。

In [2]:
# 匹配 | **key** | value | 格式的 table row
_TABLE_ROW_RE = re.compile(r'^\|\s*\*\*(.+?)\*\*\s*\|\s*(.+?)\s*\|', re.MULTILINE)


def extract_table_metadata(text: str) -> dict[str, str]:
    """從 Markdown table 中萃取 **key** → value 形式的 metadata。"""
    return {m.group(1): m.group(2).strip() for m in _TABLE_ROW_RE.finditer(text)}


doc_001 = _crm.read_file('meeting_001_晨星科技_2026-05-08.md')
meta = extract_table_metadata(doc_001)

print('萃取結果：')
for k, v in list(meta.items())[:5]:
    print(f'  {k:<8} → {v}')

萃取結果：
  客戶名稱     → 晨星科技股份有限公司
  會議日期     → 2026-05-08
  會議時間     → 14:00 – 16:00
  會議地點     → 晨星科技總部 3F 會議室 A
  紀錄人      → 王雅婷


## 2 · Section 切分（by `##` heading）

以 `## ` 開頭的行作為 section 邊界，切出各段落的 title 與 content。
忽略 `#`（文件標題）和 `###`（三層以下的小標題不獨立成 chunk）。

In [3]:
_SECTION_RE = re.compile(r'^## (.+)$', re.MULTILINE)


def split_sections(text: str) -> list[tuple[str, str]]:
    """回傳 [(section_title, section_content), ...] 清單。"""
    sections = []
    matches  = list(_SECTION_RE.finditer(text))
    for i, m in enumerate(matches):
        title = m.group(1).strip()
        start = m.end()
        end   = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        content = text[start:end].strip()
        sections.append((title, content))
    return sections


sections_001 = split_sections(doc_001)
print('meeting_001 sections:')
for i, (title, content) in enumerate(sections_001):
    lines = content.count('\n') + 1
    print(f'  [{i:02d}] {title:<24} ({lines:>3} lines)')

meeting_001 sections:
  [00] 基本資訊                     (  7 lines)
  [01] 參與者                      ( 15 lines)
  [02] 會議主題                     (  3 lines)
  [03] 客戶需求                     (  7 lines)
  [04] 客戶痛點                     (  6 lines)
  [05] 風險評估                     (  8 lines)
  [06] 決策紀錄                     ( 10 lines)
  [07] SOP 處理流程                 ( 27 lines)
  [08] 下一步行動（TODO）              (  7 lines)
  [09] 補充備註                     (  3 lines)


## 3 · MdParser：組合成完整解析器

`MdParser.parse()` 對單一文件執行：
1. 萃取 metadata（客戶名稱、日期、紀錄人）
2. 切分 sections
3. 為每個 section 產生一筆 JSONL chunk

In [4]:
@dataclass
class Chunk:
    doc_id      : str
    customer    : str
    meeting_date: str
    recorder    : str
    section     : str
    content     : str


# 不想輸出的 section（metadata 已在其他欄位、或內容無檢索價值）
_SKIP_SECTIONS = {'基本資訊', '參與者'}


class MdParser:
    def __init__(self, crm_dir: Path):
        self.crm_dir = crm_dir

    def parse(self, filename: str) -> list[Chunk]:
        """解析單一 .md 檔案，回傳 Chunk 清單。"""
        text = (self.crm_dir / filename).read_text(encoding='utf-8')
        doc_id = Path(filename).stem

        meta     = extract_table_metadata(text)
        customer = meta.get('客戶名稱', '')
        date     = meta.get('會議日期', '')
        recorder = meta.get('紀錄人', '')

        chunks = []
        for title, content in split_sections(text):
            if title in _SKIP_SECTIONS or not content.strip():
                continue
            chunks.append(Chunk(
                doc_id=doc_id,
                customer=customer,
                meeting_date=date,
                recorder=recorder,
                section=title,
                content=content,
            ))
        return chunks

    def parse_all(self, pattern: str = '*.md') -> list[Chunk]:
        """解析目錄下所有符合 pattern 的 .md 檔，回傳所有 chunks。"""
        all_chunks = []
        for fname in sorted(self.crm_dir.glob(pattern)):
            all_chunks.extend(self.parse(fname.name))
        return all_chunks

## 4 · 解析 meeting_001 — 逐筆展示

In [5]:
parser = MdParser(CRM_DIR)
chunks_001 = parser.parse('meeting_001_晨星科技_2026-05-08.md')

c0 = chunks_001[0]
print(f'doc_id      : {c0.doc_id}')
print(f'customer    : {c0.customer}')
print(f'meeting_date: {c0.meeting_date}')
print(f'recorder    : {c0.recorder}')
print(f'chunks 數量  : {len(chunks_001)}')

print()
for chunk in chunks_001[:3]:
    print(f'  section: {chunk.section}')
    preview = chunk.content[:160].replace('\n', '\n           ')
    print(f'  content: {preview}')
    if len(chunk.content) > 160:
        print('           ...')
    print()

doc_id      : meeting_001_晨星科技_2026-05-08
customer    : 晨星科技股份有限公司
meeting_date: 2026-05-08
recorder    : 王雅婷
chunks 數量  : 8

  section: 會議主題
  content: ERP 系統升級 — 初步需求討論與可行性評估
           
           ---

  section: 客戶需求
  content: 1. 將現有地端 ERP（SAP ECC 6.0）升級至雲端版本，支援多廠區統一管理。
           2. 財務模組需與現有銀行 API（玉山、國泰）串接，支援自動對帳。
           3. 新系統需支援角色型存取控制（RBAC），並通過 ISO 27001 稽核要求。
           4. 介面需提供中文化報表匯出（Excel、PDF），並相容現行 BI 工具
           ...

  section: 客戶痛點
  content: - **痛點 1｜資料孤島**：各廠區資料各自維護，彙整報表需 2–3 天人工作業。
           - **痛點 2｜稽核追蹤困難**：現有系統缺乏完整 Audit Log，法遵壓力大。
           - **痛點 3｜舊系統效能瓶頸**：月結日當天系統頻繁超時，影響財務作業。
           - **痛點 4｜人力依賴**：關鍵報表邏輯僅一位資深工程師熟悉，存
           ...



## 5 · 批次解析全部文件並輸出 JSONL

In [6]:
all_chunks = parser.parse_all()

print(f'總 chunks 數: {len(all_chunks)}')
print('按 doc 分布:')
from collections import Counter
for doc_id, count in Counter(c.doc_id for c in all_chunks).items():
    print(f'  {doc_id:<40}: {count} chunks')

# 輸出 JSONL
OUTPUT_PATH = PROJECT_ROOT / 'data' / 'crm_chunks.jsonl'
print(f'\n寫出 JSONL → data/crm_chunks.jsonl')
with OUTPUT_PATH.open('w', encoding='utf-8') as f:
    for chunk in all_chunks:
        f.write(json.dumps(asdict(chunk), ensure_ascii=False) + '\n')

print('完成 ✓')

總 chunks 數: 39
按 doc 分布:
  meeting_001_晨星科技_2026-05-08             : 8 chunks
  meeting_002_鴻圖零售_2026-05-15             : 8 chunks
  meeting_003_晨星科技_2026-05-22             : 9 chunks
  meeting_004_新星金融科技_2026-05-28           : 8 chunks
  meeting_005_晨星科技_2026-06-05             : 6 chunks

寫出 JSONL → data/crm_chunks.jsonl
完成 ✓


In [7]:
# 驗證：讀回 JSONL 並顯示前兩筆
print('JSONL 前兩筆：')
with OUTPUT_PATH.open(encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 2:
            break
        row = json.loads(line)
        # 截斷 content 顯示
        row['content'] = row['content'][:60] + ('...' if len(row['content']) > 60 else '')
        print()
        print(json.dumps(row, ensure_ascii=False))

JSONL 前兩筆：

{"doc_id": "meeting_001_晨星科技_2026-05-08", "customer": "晨星科技股份有限公司", "meeting_date": "2026-05-08", "recorder": "王雅婷", "section": "會議主題", "content": "ERP 系統升級 — 初步需求討論與可行性評估\n\n---"}

{"doc_id": "meeting_001_晨星科技_2026-05-08", "customer": "晨星科技股份有限公司", "meeting_date": "2026-05-08", "recorder": "王雅婷", "section": "客戶需求", "content": "1. 將現有地端 ERP（SAP ECC 6.0）升級至雲端版本，支援多廠區統一管理。\n2. 財務模組需與現有銀行 AP..."}


## 小結

`MdParser` 實現了以下轉換鏈：

```
.md 原始文件
  ↓ extract_table_metadata()  → customer, date, recorder
  ↓ split_sections()          → [(title, content), ...]
  ↓ Chunk dataclass           → 結構化物件
  ↓ json.dumps()              → JSONL 檔案
```

產出的 `crm_chunks.jsonl` 提供：
- **metadata 過濾**：依 `customer`、`meeting_date`、`section` 精確過濾
- **內容索引**：每個 chunk 是語意完整的段落，適合後續關鍵字搜尋
- **來源追溯**：`doc_id` 可對應回原始 .md 檔案

---
**下一個筆記本（10）**：Extract → Conflict Detection — 用 structured output 比對兩份文件，找出矛盾的決策與時程。